# Z, S, and CZ

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from examples.algorithms.diagonal_gates_tutorial import diagonal_gate_examples
from examples.algorithms.field_addition_catalog import field_signature_svg

examples = diagonal_gate_examples()

def show_signature(example):
    name, signature, _, _ = example
    display(SVG(field_signature_svg([(name, signature, {})])))

## `ZGate`
The Z gate.

This causes a phase flip: Z|+> = |-> and vice-versa.

In [ ]:
zgate = examples[0]

### Example Instances

In [ ]:
zgate[1]

#### Graphical Signature

In [ ]:
show_signature(zgate)

## `SGate`
The S gate.

The unitary matrix of `SGate` is
$$
\begin{bmatrix}
    1 & 0 \\
    0 & i
\end{bmatrix}
$$

It is the 'square root' of the Z gate: $S\cdot S = Z$.

#### Registers
 - `q`: The qubit


In [ ]:
s_gate = examples[1]

### Example Instances

In [ ]:
s_gate[1]

#### Graphical Signature

In [ ]:
show_signature(s_gate)

## `CZ`
Two-qubit controlled-Z gate.

#### Registers
 - `q1`: One-bit control register.
 - `q2`: One-bit target register.


CZ applies a minus sign only to the joint one state. Both ports are retained in their original order.

In [ ]:
cz = examples[2]

### Example Instances

In [ ]:
cz[1]

#### Graphical Signature

In [ ]:
show_signature(cz)

## Exact phase through append

Run each operation through ordinary `circuit.append(op)` and check its full complex matrix. The complete operators detect global and relative phase differences. The declared source category for each base gate is one Clifford. Each emitted example has one native opcode, with zero work wires and zero measurements.

In [ ]:
from collections import Counter
import torch
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.simulation.unitary import get_unitary

dtype = torch.complex128
expected_diagonals = {"Z": [1, -1], "S": [1, 1j], "CZ": [1, 1, 1, -1]}
for name, _, op, key in examples:
    circuit = Circuit(op.n_qubits, dtype=dtype)
    circuit.append(op)
    target = torch.diag(torch.tensor(expected_diagonals[name], dtype=dtype))
    torch.testing.assert_close(get_unitary(circuit), target, rtol=0, atol=1e-12)
    print(name, "emitted", dict(Counter(g.name for g in op.instructions)))
    print(name, "source categories", query_qec_gate_cost(key))

## Phase identities and coherent control

Two S applications produce Z, and S followed by its exact adjoint restores identity. A controlled S multiplies the joint one amplitude by i. The controlled adjoint multiplies it by −i. The controlled operations use the native circuit's exact controlled phase path, without additional scratch.

In [ ]:
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.core.ir import Instruction

op = s_gate[2]
s = Circuit(1, dtype=dtype).append(op)
inverse = Circuit(1, dtype=dtype).append(FieldOperation(1, (Instruction("sdg", (0,)),)))
twice = Circuit(1, dtype=dtype).append(op).append(op)
torch.testing.assert_close(get_unitary(twice), torch.diag(torch.tensor([1, -1], dtype=dtype)))
torch.testing.assert_close(get_unitary(inverse) @ get_unitary(s), torch.eye(2, dtype=dtype))
for block, phase in ((s, 1j), (inverse, -1j)):
    controlled = block.control(1, ctrl_qubits=(1,))
    torch.testing.assert_close(get_unitary(controlled), torch.diag(torch.tensor([1, 1, 1, phase], dtype=dtype)), rtol=0, atol=1e-12)
print("S square, adjoint and both controlled phases verified")